# Building Lattice Model Hamiltonians Tutorial
---

SAFIRE can handle model Hamiltonians of the form

$$
\hat{H} = \hat{H}_t + \hat{H}_U + \hat{H}_J,
$$
where
$$
\hat{H}_t = \sum_{\mu\nu,\sigma \sigma'}t^{\sigma \sigma'}_{\mu\nu}\hat{c}^\dagger_{\mu\sigma}\hat{c}_{\nu\sigma'},
$$

$$
\hat{H}_U =  \sum_{\mu} U_i \hat{n}_{\mu\uparrow} \hat{n}_{\mu\downarrow}
+ \sum_{\mu<\nu} U_{\mu\nu}^1 (\hat{n}_{\mu\uparrow} \hat{n}_{\nu\downarrow} + \hat{n}_{\mu\downarrow} \hat{n}_{\nu\uparrow} )
+ \sum_{\mu<\nu} U_{\mu\nu}^2 (\hat{n}_{\mu\uparrow} \hat{n}_{\nu\uparrow} + \hat{n}_{\mu\downarrow} \hat{n}_{\nu\downarrow} ),
$$

and,

$$
\hat{H}_J = \sum_{\mu<\nu} J_{\mu\nu} (
     \hat{c}^\dagger_{\mu\uparrow}\hat{c}^\dagger_{\nu\downarrow}\hat{c}_{\mu\downarrow}\hat{c}_{\nu\uparrow}
     +\hat{c}^\dagger_{\mu\uparrow}\hat{c}^\dagger_{\mu\downarrow}\hat{c}_{\nu\downarrow}\hat{c}_{\nu\uparrow}
     +\hat{c}^\dagger_{\nu\uparrow}\hat{c}^\dagger_{\mu\downarrow}\hat{c}_{\nu\downarrow}\hat{c}_{\mu\uparrow}
     +\hat{c}^\dagger_{\nu\uparrow}\hat{c}^\dagger_{\nu\downarrow}\hat{c}_{\mu\downarrow}\hat{c}_{\mu\uparrow}
     ).
$$

$\mu$ / $\nu$ are compound indices, $\mu = (i,p,m)$ / $\mu = (j,q,n)$, that include the lattice site index, ($i$ / $j$), sublattice, ($p$ / $q$) and band ($m$ / $n$) indices.
This is known as the Hubbard-Kanamori Hamiltonian.
Many standard model Hamiltonians can be written in this language including the standard Hubbard model,
extendended Hubbard with nearest-neighbor $V$, etc.
SAFIRE is agnostic to the details of the compound indices.
It simply accepts Hamiltonian "components" which are interpreted as above.
This provides great flexibilty in specifying Hamiltonians, however care must be taken outside of SAFIRE to ensure that consistent index conventions are used.

afqmctools provides a framework for building lattice model Hamiltonians that can generate broad classes of lattice model Hamiltonians
on a variety of lattices, using consistent conventions for indexing.
The framework consists of a Lattice class which is responsible for geometry (see the [Lattice class tutorial](../03_setting_up_a_lattice/03_setting_up_a_lattice.html)),
a Hamiltonian builder which is responsible for generating specific Hamiltonian terms on demand given a specific Lattice instance,
and a Hamiltonian "Director" which is responsible for choosing which build steps to perfrom based on
a set of input Hamiltonian parameters.
Each component of the framework can be used directly; however
the Director class represents the highest-level interface of the lattice Hamiltonian framework and
can manage the underlying Lattice instance and Builder instance.
Users who want direct control over the Hamiltonian build steps can directly
use the Hamiltonian "Builder". See the [Hamiltonian Builder tutorial](../05_hamiltonian_builder/05_hamiltonian_builder.html) for
more detail.
It is recommended to use the Director whenever possible.

<div>
<img src="https://users.flatironinstitute.org/~beskridge/tutorial_figs/6784ee4ea455921958ac327234b91ab07702736ab22fa2df804e8dccbc36a404/models/00_overview/HamilDirectorSystem.png" width="1000">
</div>

<b>In this tutorial, we focus on building Hamiltonian terms via the Hamiltonian "Director".</b>
We will explore the possible input parameters and their respective meanings/conventions.


# ▶️ Setup : Run me
-------------------

Run the cell below to setup a scratch directory.

In [ ]:
# setup scratch dir
from pathlib import Path
from tutorial_utils import run_afqmc, get_scratch_dir

home = Path.home()
scratch_rootdir = home / ".scratch"
scratch_rootdir.mkdir(exist_ok=True)
scratch_dir = get_scratch_dir("lattice_04_hamiltonian_director",scratch_rootdir)

# Hamiltonian Director Tutorial
---

The `afqmctools` Python package provides a framework for generating lattice model Hamiltonians as described in the Building Lattice Model Hamiltonians overview [here](#building-lattice-model-hamiltonians-tutorial).
The Hamiltonian Director is the highest-level interface to this framework.
It may be invoked as a command line tool using an input file, or a directly within a Python script.
We will briefly demonstrate the command line tool, but we will focus on using the Director within a Python script for most of this tutorial.

The Hamiltonian Director is meant to handle building Hamiltonians which can be expressed in terms of a few simple parameters on relatively simple lattices.
While SAFIRE supports any combinaiton of lattice, sublattice, and band degrees-of-freedom in the Hamiltonian, it is not always feasible to map such general models onto a few simple parameters.
For these cases, we provide lower-level classes to assist as much as possible. See the [Lattice class tutorial](../03_setting_up_a_lattice/03_setting_up_a_lattice.html) for example.
For example, the Hubbard model only needs a single parameter, $U/t$, and a lattice / boundary conditions in order to be fully specified.
More general Hamiltonians can also be built using the Director; however, it may be useful to some users to work directly with the Lattice and Hamiltonian Builder in a Python script. See the [Hamiltonian Builder tutorial](../05_hamiltonian_builder/05_hamiltonian_builder.html) for more advance uses.

In this tutorial, we will cover how to request specific Hamiltonian terms from the Hamiltonian Director.
We will also cover the details of the input conventions for each of the terms needed to build these Hamiltonians.

## Invoking the Hamiltonian Director via the CLI

---

In addition to the ability to invoke the Hamiltonian director directly within a Python script, afqmctools provides a command line tool that invokes the Hamiltonian director.
If you installed `afqmctools` using the official instructions, you can run the following to see possible options

```bash
$ make_model_ham --help

usage: make_model_ham [-h] [--input input] [--output outfile] [--free-elec] [--slater-type {closed,collinear,noncollinear}] [--plotlat] [--plot1b] [--plot2b] [--saveplots] [--pair-corr] [--hubbardU HUBBARDU]

options:
  -h, --help            show this help message and exit
  --input input, -i input
                        Input file containing Hamiltonan params (toml)
  --output outfile, -o outfile
                        Name of HDF5 file to save model Hamiltonian in
  --free-elec           Save free-electron wavefunction as well
  --slater-type {closed,collinear,noncollinear}
                        type of free-electron Slater determinant (closed,collinear,noncollinear)
  --plotlat, -pl        plot the lattice
  --plot1b              plot the 1-body Hamiltonian
  --plot2b              plot the 2-body Hamiltonian
  --saveplots, -sp      save plots instead of displaying (useful for remote connections)
  --pair-corr, -pc      Add pair-correlators to AFQMC input file (s,+x,-x,+y,-y)
  --hubbardU HUBBARDU, -U HUBBARDU
                        Hubbard U, if given and '--free-elec' is set, the hubbard U energy is evaluated using the free-electron wavefunction (Note: this is only for reference - does not effect the output wavefunction!)
```

The primary option that we are interested in here is `-i` which allows an input file (in .toml format) containing Hamiltonian parameters to be specified.
The parameter input conventions are the same whether we use the `make_model_ham` CLI tool, or use the Hamiltonian Director within a Python script.

Here is a sample input file which will generate a Hubbard model with nearest-neighbor hopping, on a 4x4 square lattice with periodic boundary conditions.

```toml
[hamiltonian]
t = 1.0
U = 4.0

[lattice]
L1 = 4
L2 = 4
boundary1 = "PBC"
boundary2 = "PBC"
```

Notice that the Hamiltonian parameters and the lattice parameters are separate.
We can then build the Hamiltonian with

```bash
make_model_ham -i input.toml
```
which generates the following output,

```bash

$ make_model_ham -i input.toml

No lattice instance supplied: building from parameters
running build step nth_neighbor_hopping((1.0,))
computing and storing 1th-nearest neighbors
Computing distance matrix of lattice
Reading lattice site positions from Lattice
computing and storing 1th-nearest image neighbors
Using same twists for up and down spins
running build step onsite_hubbard((4,))
Building Hubbard U term with U=4
Building onsite hubbard with positive U values: 4
Combining terms of the same type
Combining Hubbard U, U1, and U2 terms where possible
Max spin symmetry is  2

```

Noticate that the builder will echo what build steps are invoked using which parameters.
For example, we can see from the line,

```bash
running build step nth_neighbor_hopping((1.0,))
```

that the director called the nth_neighbor_hopping() build step with an argument of `1.0`.
We will see below that this generates nearest-neighbor hopping with t=1.0 (minus sign is implicit).
We also see that an onsite Hubbard interaction with $U = 4$ was also generated.

By default, `make_model_ham` saves the Hamiltonian in a file called `afqmc.h5`, but you can change this with the `-o [output name]` option.
For example,
```bash
make_model_ham -i input.toml -o example.h5
```
will save the Hamiltonian in `example.h5`.

We note that the in Python, the Hamiltonian Director can be given either the name of an input file or a Python dictionary containing the Hamiltonian parameters.
The Director will interpret the parameters using the same conventions regardless of which input method is used.
We refer you to the [TOML specification](https://toml.io/en/v1.0.0) for clarification on the TOML format.
For the remainder of the tutorial, we will be working directly with Python within this notebook.


## ✨ Example : Standard Hubbard Model
---------------------------------------

Below, we construct a Hubbard model on a 4x4 square lattice with both nearest, and next-nearest neighbor hopping by providing a list of values for "t".
We plot the hopping matrix to demonstrate that the Hamiltonian includes both terms.
Note that the default hopping matrix shape is $2N_{sites} \times N_{sites}$ where the first $N_{sites}$ rows correspond to the spin-up sector and the last $N_{sites}$ rows correspond to the spin-down sector.

In [ ]:
from afqmctools.systems.lattice import get_lattice
from afqmctools.hamiltonian.model.director import HamiltonianDirector
from afqmctools.utils.visualize import plot_lattice

# Define the lattice parameters
lattice_params = {
    'L1': 4,
    'L2': 4,
    'boundary1': 'PBC',
    'boundary2': 'PBC'
}

# define Hamiltonian parameters
hamiltonian_params = {
    't': 1.0,
    'U': 4.0
}

params = {
    'hamiltonian': hamiltonian_params
}

lattice = get_lattice(lattice_params)
plot_lattice(lattice)

hamiltonian = HamiltonianDirector(source=params, lattice=lattice).build()

# Writing Hamiltonians to HDF5
--------------------------------------------

In the code block below, we demonstrate how to save a lattice model Hamiltonian
to an HDF5 file that can be read by SAFIRE.
This works the same way for all Hamiltonians generated by afqmctools.
We will not repeat this below.


In [ ]:
from afqmctools.utils.io import write_model_hamiltonian

write_model_hamiltonian(
    hamiltonian=hamiltonian,
    fname=scratch_dir/"hamiltonian.h5"
)

Now, let's check the contents of the scratch directory to see that the Hamiltonian has been generated.

In [ ]:
! ls -l {scratch_dir}

# Adding more terms to the Hamiltonian
--------------------------------------

## Adding Hopping

---

The Hamiltonian Director supports nth-order neighbor hopping and band-dependent hopping.
$n$ is limited by the condition that sites are not allowed to be their own neighbors via
periodic boundary conditions.
In this case, an error will be raised.

The simplest case,
nearest-neighbor hopping with $t$ = 1, is included in the Hamiltonian by default.
For multiband Hamiltonians, the hopping is uniform across bands and it is assumed that
there is no intrasite hopping.
This is handles by a different Hamiltonian term, which we refer to as $\epsilon$.
You can change this value by providing a different value of $t$ as

```toml
[hamiltonian]
t = 0.5
```

or as a Python dictionary

```python
params = {
    'Hamiltonian' : {
        't' : 0.5
    }    
}
```

hopping can be turned off by either setting $t$ to 0, or (in Python only) setting t to `None`.

nth-order neighbor hopping can be included by setting $t$ to a list of length $n$
where the nth-entry is interpreted as the hopping strength for nth-order neighbors.
Formally, if $t_n$ are the elements of the list, $t$,
the following term will be added,
$$
\hat{H}_t = \sum_n - t_n \sum_{\langle i,j \rangle^n} \hat{c}^\dagger_i \hat{c}_j
$$
where $\langle i,j \rangle^n$ indicates that the sum is constrained to run over nth-order neighbors.

> **NOTE**:
> nth-order neighbors are determined by spatial distance. This is especially important to keep in mind when attempting to use a lattice with a basis.

<div class="alert alert-block alert-warning">
<b>CAUTION:</b>
    nth-order neighbors are determined by spatial distance. This is especially important to keep in mind when attempting to use a lattice with a basis.
</div>

## ✨ Example : Hubbard Model with $t^{`}$
--------------------------------------

Below, we construct a Hubbard model on a 4x4 square lattice with both nearest, and next-nearest neighbor hopping by providing a list of values for "t".
We plot the hopping matrix to demonstrate that the Hamiltonian includes both terms.
Note that the default hopping matrix shape is $2N_{sites} \times N_{sites}$ where the first $N_{sites}$ rows correspond to the spin-up sector and the last $N_{sites}$ rows correspond to the spin-down sector.

In [ ]:
import matplotlib.pyplot as plt

from afqmctools.systems.lattice import get_lattice
from afqmctools.hamiltonian.model.director import HamiltonianDirector
from afqmctools.utils.visualize import plot_lattice

# define Hamiltonian parameters
params = {
    'lattice': {
        'L1': 4,
        'L2': 4,
        'boundary1': 'PBC',
        'boundary2': 'PBC'
    },
    'hamiltonian': {
        'U': 4.0,
        't' : [1.0,0.5] # implicit minus sign!
    }
}

# this will make a Hubbard model on a 4x4 lattice with PBCs
hamiltonian = HamiltonianDirector(source=params).build()

hopping = hamiltonian.get_one_body()
plt.matshow(hopping.toarray())
plt.colorbar()
plt.show()

## Adding onsite Hubbard U

---


SAFIRE supports arbitrary onsite interactions of the from

$$
\hat{H}_U =  \sum_{\mu} U_{\mu} \hat{n}_{\mu\uparrow} \hat{n}_{\mu\downarrow},
$$

where, again, the combined index $\mu$ may include both lattice site and band indices.
We've already seen how to add a basic nearest-neighbor onsite Hubbard U, now we see see how to provide a site-, and/or band-dependent onsite interaction.
<b>This input parameter, $U$, is in general an array. The band/site dependence of the interaction is deduced based on the shape of the $U$ array</b>.

### Band- / site-dependence deduction rules

---

1. if U is a scalar (or an array with a single entry), then the onsite Hubbard interaction term is included with strength U and applied to all sites and bands unformly.

2. if U is 1-dimensional with length nbands (i.e. [U1,U2,…,Um] where for an m-band model), then the onsite Hubbard interaction term is included with strength U_i applied to band i and uniformaly across sites.

3. if U is 1-dimensional with length nsites (i.e. [U1,U2,…,Un] where n is the number of sites), then the onsite Hubbard interaction term is included with strength U_i applied to site i and uniformly across bands.

4. If nbands and nsites are equal, then nbands take priority (i.e. rule 2 applies and not rule 3).


## ✨ Example: site-dependent U
-------------------------------

In the example below, we build a site-dependent U with a checkerboard pattern.

In [ ]:
from afqmctools.systems.lattice import get_lattice
from afqmctools.hamiltonian.model.director import HamiltonianDirector
from afqmctools.utils.visualize import plot_lattice

import numpy as np

lattice_params = {
    'L1': 2,
    'L2': 4,
    'boundary1': 'PBC',
    'boundary2': 'PBC'
}

lattice = get_lattice(lattice_params)


# define Hamiltonian parameters
hamiltonian_params = {
    'nbands' : 1,
    'U': [1.0,2.0,1.0,2.0,2.0,1.0,2.0,1.0]
}

params = {
    'hamiltonian': hamiltonian_params
}

# this will make a Hubbard model on a 4x4 lattice with PBCs
hamiltonian = HamiltonianDirector(source=params, lattice=lattice).build()

# let's take a look at the U term:
U = hamiltonian.get_U().toarray().diagonal().reshape(lattice.L)

plot_lattice(
    lattice,
    density=U,
    vmin=0.0,
    vmax=3.0
)

## ✨ Example: band-dependent U
----

Now we will construct a band-dependent U which is *uniform* accross lattice sites.
We will use two bands and the first band will have $U/t = 1$
and the second will have $U/t = 2$.

As a bonus, we also demonstrate how to include a band-dependent hopping.

In [ ]:
from afqmctools.systems.lattice import get_lattice
from afqmctools.hamiltonian.model.director import HamiltonianDirector
from afqmctools.utils.visualize import plot_lattice

import numpy as np

lattice_params = {
    'L1': 2,
    'L2': 4,
    'boundary1': 'PBC',
    'boundary2': 'PBC'
}
lattice = get_lattice(lattice_params)


# define Hamiltonian parameters
hamiltonian_params = {
    'nbands' : 2,
    't' : [np.array([[1.0, -0.2], [-0.2, 1.0]])], # bonus: band-dependent hopping
    'U': [1.0,2.0]
}

params = {
    'hamiltonian': hamiltonian_params
}

# this will make a Hubbard model on a 4x4 lattice with PBCs
hamiltonian = HamiltonianDirector(source=params, lattice=lattice).build()

# let's take a look at the U term:
U_shape = (lattice.L[0],lattice.L[1],hamiltonian_params['nbands'])
U = hamiltonian.get_U().toarray().diagonal().reshape(U_shape)
plot_lattice(
    lattice,
    density=U[:,:,0],
    title='U for band 0',
    vmin=0.0,
    vmax=3.0
)
plot_lattice(
    lattice,
    density=U[:,:,1],
    title='U for band 1',
    vmin=0.0,
    vmax=3.0
)

In [ ]:
from afqmctools.systems.lattice import get_lattice
from afqmctools.hamiltonian.model.director import HamiltonianDirector
from afqmctools.utils.visualize import plot_lattice

import numpy as np

lattice_params = {
    'L1': 2,
    'L2': 4,
    'boundary1': 'PBC',
    'boundary2': 'PBC'
}
lattice = get_lattice(lattice_params)


# define Hamiltonian parameters
hamiltonian_params = {
    'nbands' : 2,
    't' : [np.array([[1.0, -0.2], [-0.2, 1.0]])], # bonus: band-dependent hopping
    'U': [1.0,2.0]
}

params = {
    'hamiltonian': hamiltonian_params
}

# this will make a Hubbard model on a 4x4 lattice with PBCs
hamiltonian = HamiltonianDirector(source=params, lattice=lattice).build()

# let's take a look at the U term:
U_shape = (lattice.L[0],lattice.L[1],hamiltonian_params['nbands'])
U = hamiltonian.get_U().toarray().diagonal().reshape(U_shape)
plot_lattice(
    lattice,
    density=U[:,:,0],
    title='U for band 0',
    vmin=0.0,
    vmax=3.0
)
plot_lattice(
    lattice,
    density=U[:,:,1],
    title='U for band 1',
    vmin=0.0,
    vmax=3.0
)

## Adding Density-Density interaction
-------------------------------------

SAFIRE supports general interactions of the form,

$$
\hat{H}_{U1} =  \sum_{\mu<\nu} U_{\mu\nu}^1 (\hat{n}_{\mu\uparrow} \hat{n}_{\nu\downarrow} + \hat{n}_{\mu\downarrow} \hat{n}_{\nu\uparrow} ),
$$

where $\mu = (i,m)$ / $\nu = (i,p)$ are a composite lattice, $i$ / $j$, and band $m$, $p$ indices.

afqmctools is able to generate a subset of interactions of this type;
The $U_{\mu\nu}^1$ matrix is constructed based on what information
the user provides.
specifically, it handles the following cases.


### Case 1: Uniform intrasite interaction
-----------------------------------------

The user provides a single value for $U^1$ and keeps the default $n=0$.

In this case, the density-density intercation takes the following form.

$$
\hat{H}_{U1} = U^1 \sum_{i} \sum_{mp} (\hat{n}_{(i,m)\uparrow} \hat{n}_{(i,p)\downarrow} + \hat{n}_{(i,m)\downarrow} \hat{n}_{(i,p)\uparrow} ),
$$

where we have replaced the composite indicies with site and band indices for clarity.


### Case 2: Band-dependent intrasite interaction
------------------------------------------------

The user provides a matrix with shape $nbands \times nbands$,

$$
U^1 = U^1_{mp},
$$

and keeps the default $n = 0$ for the neighbor order parameter.

In this case, the density-density intercation takes the following form.

$$
\hat{H}_{U1} =  \sum_{i} \sum_{mp} U_{mp}^1 (\hat{n}_{(i,m)\uparrow} \hat{n}_{(i,p)\downarrow} + \hat{n}_{(i,m)\downarrow} \hat{n}_{(i,p)\uparrow} ),
$$

where we have replaced the composite indicies with site and band indices for clarity.

### Case 3: Uniform nth-order neighbor interaction
--------------------------------------------------


### Case 4: Band-dependent nth-order neighbor interaction
----------------------------------------------------------------

The user provides the neighor order, $n$, and a
 matrix with shape $nbands \times nbands$,

$$
U^1 = U^1_{mp}.
$$

In this case, the density-density intercation takes the following form.

$$
\hat{H}_{U1} =  \sum_{\langle i,j \rangle^n} \sum_{mp} U_{mp}^1 (\hat{n}_{(i,m)\uparrow} \hat{n}_{(i,p)\downarrow} + \hat{n}_{(i,m)\downarrow} \hat{n}_{(i,p)\uparrow} ),
$$

where we have replaced the composite indicies with site and band indices for clarity.

### Case 5: a list of inputs from the cases above
-------------------------------------------------

If a user provides a list of valid inputs, then afqmctools will generate each
term provided <b>using the list index / position for the neighbor order "$n$."</b>
For example, if a user provides

```toml
[hamiltonian]
nbands = 2
U1= [ 2.0,
      [[ 0.5, 1.0 ],
       [ 1.0, 0.5 ]]
    ]
```

Both a uniform intrasite interaction with U1 = 2.0 (n=0 / onsite),
and a band-dependent nearest-neighbor (n=1) interaction will be added to the
Hamiltonian.


## Adding Spin-Spin interaction
-------------------------------------

SAFIRE supports general interactions of the form,

$$
\hat{H}_{U2} = \sum_{\mu<\nu} U_{\mu\nu}^2 (\hat{n}_{\mu\uparrow} \hat{n}_{\nu\uparrow} + \hat{n}_{\mu\downarrow} \hat{n}_{\nu\downarrow} ),
$$

where $\mu = (i,m)$ / $\nu = (i,p)$ are a composite lattice, $i$ / $j$, and band $m$, $p$ indices.

afqmctools is able to generate a subset of interactions of this type;
The $U_{\mu\nu}^2$ matrix is constructed based on what information
the user provides simimilarly to the density-density (i.e. the $U^1$) term.
See the density-density section above for more details.


## Adding a Hund's J term
-------------------------

SAFIRE supports general interactions of the form,

$$
\hat{H}_J = \sum_{\mu<\nu} J_{\mu\nu} (
     \hat{c}^\dagger_{\mu\uparrow}\hat{c}^\dagger_{\nu\downarrow}\hat{c}_{\mu\downarrow}\hat{c}_{\nu\uparrow}
     +\hat{c}^\dagger_{\mu\uparrow}\hat{c}^\dagger_{\mu\downarrow}\hat{c}_{\nu\downarrow}\hat{c}_{\nu\uparrow}
     +\hat{c}^\dagger_{\nu\uparrow}\hat{c}^\dagger_{\mu\downarrow}\hat{c}_{\nu\downarrow}\hat{c}_{\mu\uparrow}
     +\hat{c}^\dagger_{\nu\uparrow}\hat{c}^\dagger_{\nu\downarrow}\hat{c}_{\mu\downarrow}\hat{c}_{\mu\uparrow}
     ).
$$

$\mu$ / $\nu$ are compound indices, $\mu = (i,p,m)$ / $\mu = (j,q,n)$, that include the lattice site index, ($i$ / $j$), sublattice, ($p$ / $q$) and band ($m$ / $n$) indices.

afqmctools is able to generate a subset of interactions of this type;
The $J_{\mu\nu}$ matrix is constructed based on what information
the user provides simimilarly to the density-density (i.e. the $U^1$ term).
See the density-density section above for more details.

## ✨ Example: 3-band Hubbard-Kanamori Hamiltonian
--------------------------------------------------

Below we construct a 3-band Hubbard kanamor model using uniform values for U,U1,U2, and J as in Table I of [PRB 99, 235142 (2019)](https://doi.org/10.1103/PhysRevB.99.235142)


In [ ]:
from afqmctools.systems.lattice import get_lattice
from afqmctools.hamiltonian.model.director import HamiltonianDirector
from afqmctools.utils.visualize import plot_lattice

# Define the lattice parameters
lattice_params = {
    'L1': 6,
    'L2': 1,
    'boundary1': 'PBC',
    'boundary2': 'open'
}

# define Hamiltonian parameters
hamiltonian_params = {
    'nbands' : 3,
    't' : 1.0,
    'U': 4.0,
    'U1': 1.5,
    'U2': 1.0,
    'J': 0.5
}

params = {
    'hamiltonian': hamiltonian_params
}

lattice = get_lattice(lattice_params)
plot_lattice(lattice)

hamiltonian = HamiltonianDirector(source=params, lattice=lattice).build()

## Adding an extended Hubbard V interaction
-------------------------------------------

The $n^{th}$-order neighbor extended Hubbard interaction,

$$
\hat{H}_V = \sum_{\langle i<j \rangle^n} \sum_{\sigma \sigma'} V_{\mu \nu} \hat{n}_{\mu \sigma } \hat{n}_{\nu \sigma'},
$$

where $\mu = (i,m)$ / $\nu = (i,p)$ are a composite lattice, $i$ / $j$, and band $m$, $p$ indices, can be expressed in terms of $\hat{H}_{U1}$ and $\hat{H}_{U2}$.

$$
\hat{H}_V = \sum_{\langle i<j \rangle^n} V_{\mu \nu} (\hat{n}_{\mu \uparrow} \hat{n}_{\nu \downarrow} + \hat{n}_{\mu \downarrow} \hat{n}_{\nu \uparrow})
+ \sum_{\langle i<j \rangle^n} V_{\mu \nu} (\hat{n}_{\mu \uparrow} \hat{n}_{\nu \uparrow} + \hat{n}_{\mu \downarrow} \hat{n}_{\nu \downarrow}),
$$

which we can identify as,

$$
\hat{H}_V = \hat{H}_{U1}[U1 = V] + \hat{H}_{U2}[U2 = V],
$$

for $n^{th}$-order neighbors.e

afqmctools is able to generate a subset of interactions of this type using the
same input conventions as for $\hat{H}_{U1}$, and $\hat{H}_{U2}$.



## ✨ Example: Extended Hubbard Model
-------------------------------------

Below, we build an extended Hubbard model including nearest-neighbor V.

If you inspect the output, you will see that behind the scenes, we use a U1 and U2 term to represent the extended Hubbard V since those terms are directly implemented in SAFIRE.

In [ ]:
import matplotlib.pyplot as plt

from afqmctools.systems.lattice import get_lattice
from afqmctools.hamiltonian.model.director import HamiltonianDirector
from afqmctools.utils.visualize import plot_lattice

# define Hamiltonian parameters
params = {
    'lattice': {
        'L1': 4,
        'L2': 4,
        'boundary1': 'PBC',
        'boundary2': 'PBC'
    },
    'hamiltonian': {
        't' : 1.0,
        'U': 4.0,
        'V' : 1.0
    }
}

hamiltonian = HamiltonianDirector(source=params).build()